# Vision Transformer - OrganCMNIST
**CAB420 Assessment 2**

This notebook implements a compact custom Vision Transformer (ViT) trained on the OrganCMNIST dataset from MedMNIST. The model is trained across three random seeds to report stable mean and standard deviation results.

**Architecture summary:**
- Input: 28 × 28 × 1 grayscale CT images (native resolution, no resizing)
- Patch size: 4 × 4 → 49 patches per image
- 4 transformer blocks, 4 attention heads, projection dim 64
- Training-only brightness/contrast augmentation (no horizontal flips due to laterality labels)
- Three seeds: 42, 123, 2025

## 1. Imports

In [ ]:
import os
import time
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
import medmnist
from medmnist import INFO
from medmnist.dataset import OrganCMNIST

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

## 2. Configuration

All hyperparameters and output paths are defined here so they are easy to find and change in one place.

In [ ]:
# Main settings for the final ViT runs.
# I am using three seeds so the result is not based on one lucky run.
SEEDS = [42, 123, 2025]
DATASET_NAME = "organcmnist"

RUN_NAME = "Vision Transformer"

# Keeping this in its own output folder so it does not overwrite or get mixed up with the earlier compact runs.
OUTPUT_DIR = "outputs"
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
RUNS_DIR = os.path.join(OUTPUT_DIR, "runs_by_seed")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

print(f"Output folder for this run: {OUTPUT_DIR}")


# Model settings. OrganCMNIST is 28x28
IMAGE_SIZE = 28
PATCH_SIZE = 4
NUM_PATCHES = (IMAGE_SIZE // PATCH_SIZE) ** 2

PROJECTION_DIM = 64
TRANSFORMER_LAYERS = 4
NUM_HEADS = 4
KEY_DIM = PROJECTION_DIM // NUM_HEADS
MLP_DIM = 128
DROPOUT_RATE = 0.1
LEARNING_RATE = 0.001

BATCH_SIZE = 64
EPOCHS = 30
EARLY_STOPPING_PATIENCE = 5
APPLY_AUGMENTATION = True

print(f"Patch size: {PATCH_SIZE}x{PATCH_SIZE} → {NUM_PATCHES} patches per image")
print(f"Projection dim: {PROJECTION_DIM}, Layers: {TRANSFORMER_LAYERS}, Heads: {NUM_HEADS}")
print(f"Augmentation: {APPLY_AUGMENTATION}")

## 3. Dataset Info

In [ ]:
# Get the MedMNIST dataset details so the labels and class count are not hard coded.
info = INFO[DATASET_NAME]
class_names = info["label"]
num_classes = len(class_names)
target_names = [class_names[str(i)] for i in range(num_classes)]

print("TensorFlow version:", tf.__version__)
print("MedMNIST version:", medmnist.__version__)
print("\nDataset:", DATASET_NAME)
print("Task:", info["task"])
print("Number of channels:", info["n_channels"])
print("Number of classes:", num_classes)
print("Samples:", info["n_samples"])
print("Class labels:", class_names)

## 4. Data Loading and Preprocessing

In [ ]:
def load_organcmnist_data():
    """Load OrganCMNIST and do the small amount of preprocessing needed for TensorFlow."""
    train_dataset = OrganCMNIST(split="train", download=True)
    val_dataset = OrganCMNIST(split="val", download=True)
    test_dataset = OrganCMNIST(split="test", download=True)

    x_train = train_dataset.imgs
    y_train = train_dataset.labels.squeeze()

    x_val = val_dataset.imgs
    y_val = val_dataset.labels.squeeze()

    x_test = test_dataset.imgs
    y_test = test_dataset.labels.squeeze()

    print("\nOriginal data shapes:")
    print("x_train:", x_train.shape)
    print("y_train:", y_train.shape)
    print("x_val:", x_val.shape)
    print("y_val:", y_val.shape)
    print("x_test:", x_test.shape)
    print("y_test:", y_test.shape)

    # The images are already 28x28. This only scales pixels from 0-255 to 0-1.
    x_train = x_train.astype("float32") / 255.0
    x_val = x_val.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0

    # Add the grayscale channel dimension: (N, 28, 28) -> (N, 28, 28, 1).
    if x_train.ndim == 3:
        x_train = np.expand_dims(x_train, axis=-1)
        x_val = np.expand_dims(x_val, axis=-1)
        x_test = np.expand_dims(x_test, axis=-1)

    y_train = y_train.astype("int32")
    y_val = y_val.astype("int32")
    y_test = y_test.astype("int32")

    print("\nPreprocessed data shapes:")
    print("x_train:", x_train.shape)
    print("y_train:", y_train.shape)
    print("x_val:", x_val.shape)
    print("y_val:", y_val.shape)
    print("x_test:", x_test.shape)
    print("y_test:", y_test.shape)

    return x_train, y_train, x_val, y_val, x_test, y_test


x_train, y_train, x_val, y_val, x_test, y_test = load_organcmnist_data()

## 5. Augmentation and tf.data Pipeline

Brightness and contrast jitter are applied to training images only. Horizontal flips are deliberately excluded because OrganCMNIST contains laterality-based classes (kidney-left/right, femur-left/right) where flipping would create incorrect labels.

In [ ]:
# Small intensity augmentation for training only.
# I do not use flips because the dataset has left/right labels, so flipping can make labels wrong.
def augment_image(image, label):
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label


def make_tf_datasets(x_train, y_train, x_val, y_val, x_test, y_test, seed):
    train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
    val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val))
    test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))

    train_ds = train_ds.shuffle(buffer_size=len(x_train), seed=seed)

    if APPLY_AUGMENTATION:
        train_ds = train_ds.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)

    train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    return train_ds, val_ds, test_ds

## 6. Model Definition

### 6.1 Patch Extraction and Encoding

Each image is divided into non-overlapping 4×4 patches. The patches are then projected into a 64-dimensional embedding space and combined with learned positional embeddings so the model knows where each patch came from.

In [ ]:
@tf.keras.utils.register_keras_serializable()
class Patches(tf.keras.layers.Layer):
    """Split each image into non-overlapping patches."""

    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches

    def get_config(self):
        config = super().get_config()
        config.update({"patch_size": self.patch_size})
        return config


@tf.keras.utils.register_keras_serializable()
class PatchEncoder(tf.keras.layers.Layer):
    """Project patches into an embedding space and add positional information."""

    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim

        self.projection = tf.keras.layers.Dense(units=projection_dim)
        self.position_embedding = tf.keras.layers.Embedding(
            input_dim=num_patches,
            output_dim=projection_dim,
        )

    def build(self, input_shape):
        self.projection.build(input_shape)
        self.position_embedding.build((self.num_patches,))
        super().build(input_shape)

    def call(self, patches):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        return self.projection(patches) + self.position_embedding(positions)

    def get_config(self):
        config = super().get_config()
        config.update({
            "num_patches": self.num_patches,
            "projection_dim": self.projection_dim,
        })
        return config

### 6.2 Transformer Encoder and Full ViT Model

Each transformer block applies layer normalisation, multi-head self-attention, a residual connection, and a feed-forward MLP with GELU activation. Global average pooling is used instead of a CLS token to aggregate the patch representations before classification.

In [ ]:
def mlp_block(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = tf.keras.layers.Dense(units, activation=tf.nn.gelu)(x)
        x = tf.keras.layers.Dropout(dropout_rate)(x)
    return x


def build_vit_model():
    inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 1))

    patches = Patches(PATCH_SIZE)(inputs)
    encoded_patches = PatchEncoder(NUM_PATCHES, PROJECTION_DIM)(patches)

    x = encoded_patches

    for _ in range(TRANSFORMER_LAYERS):
        # Attention block.
        x1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
        attention_output = tf.keras.layers.MultiHeadAttention(
            num_heads=NUM_HEADS,
            key_dim=KEY_DIM,
            dropout=DROPOUT_RATE,
        )(x1, x1)
        x2 = tf.keras.layers.Add()([attention_output, x])

        # Feed-forward block.
        x3 = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = mlp_block(
            x3,
            hidden_units=[MLP_DIM, PROJECTION_DIM],
            dropout_rate=DROPOUT_RATE,
        )
        x = tf.keras.layers.Add()([x3, x2])

    representation = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    representation = tf.keras.layers.GlobalAveragePooling1D()(representation)
    representation = tf.keras.layers.Dropout(DROPOUT_RATE)(representation)

    features = mlp_block(
        representation,
        hidden_units=[MLP_DIM],
        dropout_rate=DROPOUT_RATE,
    )

    logits = tf.keras.layers.Dense(num_classes)(features)
    model = tf.keras.Model(inputs=inputs, outputs=logits)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"],
    )

    return model


# Quick check: build a model and print parameter count.
_test_model = build_vit_model()
print("Total trainable parameters:", _test_model.count_params())
del _test_model
tf.keras.backend.clear_session()

## 7. Plotting Functions

In [ ]:
def plot_training_curves(history_df, figure_dir):
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["accuracy"], label="Training accuracy")
    plt.plot(history_df["val_accuracy"], label="Validation accuracy")
    plt.title("Vision Transformer Accuracy During Training")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(figure_dir, "vit_accuracy_curve.png"), dpi=300)
    plt.show()
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["loss"], label="Training loss")
    plt.plot(history_df["val_loss"], label="Validation loss")
    plt.title("Vision Transformer Loss During Training")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(figure_dir, "vit_loss_curve.png"), dpi=300)
    plt.show()
    plt.close()


def plot_confusion_matrices(cm, cm_normalised, figure_dir):
    tick_marks = np.arange(num_classes)

    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation="nearest")
    plt.title("Vision Transformer Confusion Matrix")
    plt.colorbar()
    plt.xticks(tick_marks, target_names, rotation=45, ha="right")
    plt.yticks(tick_marks, target_names)
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.savefig(os.path.join(figure_dir, "vit_confusion_matrix.png"), dpi=300)
    plt.show()
    plt.close()

    plt.figure(figsize=(10, 8))
    plt.imshow(cm_normalised, interpolation="nearest", vmin=0, vmax=1)
    plt.title("Vision Transformer Normalised Confusion Matrix")
    plt.colorbar()
    plt.xticks(tick_marks, target_names, rotation=45, ha="right")
    plt.yticks(tick_marks, target_names)

    for i in range(num_classes):
        for j in range(num_classes):
            plt.text(
                j,
                i,
                f"{cm_normalised[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=7,
            )

    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.savefig(
        os.path.join(figure_dir, "vit_confusion_matrix_normalised.png"),
        dpi=300,
    )
    plt.show()
    plt.close()

## 8. Single Seed Training and Evaluation

This function handles one complete training run for a given seed: builds the model, trains with early stopping, reloads the best checkpoint, evaluates on the test set, and saves all outputs.

In [ ]:
def run_single_seed(seed, x_train, y_train, x_val, y_val, x_test, y_test):
    print("\n" + "=" * 70)
    print(f"Starting run for seed {seed}")
    print("=" * 70)

    np.random.seed(seed)
    tf.random.set_seed(seed)

    seed_dir = os.path.join(RUNS_DIR, f"seed_{seed}")
    seed_results_dir = os.path.join(seed_dir, "results")
    seed_figure_dir = os.path.join(seed_dir, "figures")
    seed_model_dir = os.path.join(seed_dir, "models")

    os.makedirs(seed_results_dir, exist_ok=True)
    os.makedirs(seed_figure_dir, exist_ok=True)
    os.makedirs(seed_model_dir, exist_ok=True)

    best_model_path = os.path.join(
        seed_model_dir,
        f"vit_organcmnist_final_augmented_seed_{seed}_best.keras",
    )

    train_ds, val_ds, test_ds = make_tf_datasets(
        x_train, y_train, x_val, y_val, x_test, y_test, seed,
    )

    model = build_vit_model()
    print("\nModel parameters:", model.count_params())
    print("Training augmentation:", "brightness/contrast" if APPLY_AUGMENTATION else "none")

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=best_model_path,
            monitor="val_loss",
            save_best_only=True,
        ),
    ]

    print("\nTraining ViT...")
    start_train_time = time.time()

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
    )

    training_time = time.time() - start_train_time

    print("\nTraining completed.")
    print(f"Training time: {training_time:.2f} seconds")
    print(f"Best model saved to: {best_model_path}")

    # Reload the checkpoint so the evaluation is based on the saved model file.
    best_model = tf.keras.models.load_model(
        best_model_path,
        custom_objects={
            "Patches": Patches,
            "PatchEncoder": PatchEncoder,
        },
    )
    print("Saved model reloaded successfully.")

    history_df = pd.DataFrame(history.history)
    history_df.to_csv(
        os.path.join(seed_results_dir, "vit_training_history.csv"),
        index=False,
    )
    plot_training_curves(history_df, seed_figure_dir)

    print("\nEvaluating on the test set...")
    start_test_time = time.time()
    test_logits = best_model.predict(test_ds)
    test_time = time.time() - start_test_time

    y_pred = np.argmax(test_logits, axis=1)

    test_accuracy = accuracy_score(y_test, y_pred)
    test_macro_f1 = f1_score(y_test, y_pred, average="macro")
    test_weighted_f1 = f1_score(y_test, y_pred, average="weighted")

    print("\nTest results:")
    print(f"Seed: {seed}")
    print(f"Test accuracy: {test_accuracy:.4f}")
    print(f"Macro F1: {test_macro_f1:.4f}")
    print(f"Weighted F1: {test_weighted_f1:.4f}")
    print(f"Test inference time: {test_time:.2f} seconds")
    print(f"Parameters: {best_model.count_params()}")

    report_dict = classification_report(
        y_test, y_pred, target_names=target_names, output_dict=True, zero_division=0,
    )
    pd.DataFrame(report_dict).transpose().to_csv(
        os.path.join(seed_results_dir, "vit_classification_report.csv")
    )

    print("\nClassification report:")
    print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))

    cm = confusion_matrix(y_test, y_pred)
    cm_normalised = cm.astype("float32") / cm.sum(axis=1, keepdims=True)

    pd.DataFrame(cm, index=target_names, columns=target_names).to_csv(
        os.path.join(seed_results_dir, "vit_confusion_matrix.csv")
    )
    pd.DataFrame(cm_normalised, index=target_names, columns=target_names).to_csv(
        os.path.join(seed_results_dir, "vit_confusion_matrix_normalised.csv")
    )
    plot_confusion_matrices(cm, cm_normalised, seed_figure_dir)

    model_config = pd.DataFrame([{
        "seed": seed,
        "dataset": DATASET_NAME,
        "image_size": IMAGE_SIZE,
        "channels": 1,
        "num_classes": num_classes,
        "patch_size": PATCH_SIZE,
        "num_patches": NUM_PATCHES,
        "projection_dim": PROJECTION_DIM,
        "transformer_layers": TRANSFORMER_LAYERS,
        "num_heads": NUM_HEADS,
        "key_dim_per_head": KEY_DIM,
        "mlp_dim": MLP_DIM,
        "dropout_rate": DROPOUT_RATE,
        "learning_rate": LEARNING_RATE,
        "augmentation": "brightness_contrast" if APPLY_AUGMENTATION else "none",
        "batch_size": BATCH_SIZE,
        "max_epochs": EPOCHS,
        "epochs_ran": len(history.history["loss"]),
        "early_stopping_monitor": "val_loss",
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "parameters": best_model.count_params(),
    }])
    model_config.to_csv(
        os.path.join(seed_results_dir, "vit_model_config.csv"),
        index=False,
    )

    run_result = {
        "seed": seed,
        "test_accuracy": test_accuracy,
        "macro_f1": test_macro_f1,
        "weighted_f1": test_weighted_f1,
        "training_time_seconds": training_time,
        "test_inference_time_seconds": test_time,
        "parameters": best_model.count_params(),
        "epochs_ran": len(history.history["loss"]),
        "best_val_accuracy": max(history.history["val_accuracy"]),
        "best_val_loss": min(history.history["val_loss"]),
        "model_path": best_model_path,
    }

    pd.DataFrame([run_result]).to_csv(
        os.path.join(seed_results_dir, "vit_seed_result.csv"),
        index=False,
    )

    print(f"\nSeed {seed} completed successfully.")

    # Clear these before the next seed so memory does not creep up.
    del model
    del best_model
    del train_ds
    del val_ds
    del test_ds
    gc.collect()
    tf.keras.backend.clear_session()

    return run_result

## 9. Run All Seeds

Trains the ViT three times with seeds 42, 123 and 2025. Each run saves its own outputs under `outputs/runs_by_seed/seed_X/`.

In [ ]:
all_results = []
for seed in SEEDS:
    result = run_single_seed(
        seed, x_train, y_train, x_val, y_val, x_test, y_test,
    )
    all_results.append(result)

## 10. Repeated Run Summary

Aggregate results across all three seeds and save the summary CSVs used in the report.

In [ ]:
results_df = pd.DataFrame(all_results)

repeated_results_path = os.path.join(RESULTS_DIR, "vit_repeated_runs_results.csv")
results_df.to_csv(repeated_results_path, index=False)

print("\n" + "=" * 70)
print("Repeated runs completed")
print("=" * 70)
print("\nIndividual run results:")
print(results_df.to_string(index=False))

metric_columns = [
    "test_accuracy",
    "macro_f1",
    "weighted_f1",
    "training_time_seconds",
    "test_inference_time_seconds",
    "epochs_ran",
]

summary_rows = []
for metric in metric_columns:
    summary_rows.append({
        "metric": metric,
        "mean": results_df[metric].mean(),
        "std": results_df[metric].std(),
    })

summary_df = pd.DataFrame(summary_rows)
repeated_summary_path = os.path.join(RESULTS_DIR, "vit_repeated_runs_summary.csv")
summary_df.to_csv(repeated_summary_path, index=False)

print("\nRepeated run summary (mean ± std):")
print(summary_df.to_string(index=False))

group_summary = pd.DataFrame([{
    "method": "Vision Transformer",
    "test_accuracy_mean": results_df["test_accuracy"].mean(),
    "test_accuracy_std": results_df["test_accuracy"].std(),
    "macro_f1_mean": results_df["macro_f1"].mean(),
    "macro_f1_std": results_df["macro_f1"].std(),
    "weighted_f1_mean": results_df["weighted_f1"].mean(),
    "weighted_f1_std": results_df["weighted_f1"].std(),
    "training_time_seconds_mean": results_df["training_time_seconds"].mean(),
    "training_time_seconds_std": results_df["training_time_seconds"].std(),
    "test_inference_time_seconds_mean": results_df["test_inference_time_seconds"].mean(),
    "test_inference_time_seconds_std": results_df["test_inference_time_seconds"].std(),
    "parameters": int(results_df["parameters"].iloc[0]),
    "seeds": ", ".join(str(seed) for seed in SEEDS),
    "notes": "Mean and standard deviation across three runs using compact custom ViT with training-only brightness/contrast augmentation",
}])

group_summary_path = os.path.join(RESULTS_DIR, "vit_group_summary_row.csv")
group_summary.to_csv(group_summary_path, index=False)

print("\nSaved repeated run outputs:")
print(f"- {repeated_results_path}")
print(f"- {repeated_summary_path}")
print(f"- {group_summary_path}")
print(f"- Per-seed outputs saved under {RUNS_DIR}")
print("\nAll repeated runs completed successfully.")